In [1]:
%autosave 0

Autosave disabled


In [2]:
# !pip install grpcio tensorflow-serving-api==2.10.1


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [9]:
import sys
!{sys.executable} -m pip install -U tensorflow-serving-api



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/conda/bin/python -m pip install --upgrade pip


In [14]:
import sys
!{sys.executable} -m pip install -U keras-image-helper

  Using cached keras_image_helper-0.0.2-py3-none-any.whl.metadata (3.5 kB)
Using cached keras_image_helper-0.0.2-py3-none-any.whl (5.4 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/conda/bin/python -m pip install --upgrade pip


In [3]:
# !pip install keras-image-helper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 53.2 MB/s eta 0:00:000m eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.1
    Uninstalling numpy-2.3.1:
      Successfully uninstalled numpy-2.3.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [keras-image-helper]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [7]:
import sys
!{sys.executable} -m pip install -U numpy

  Using cached numpy-2.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.4 MB)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/conda/bin/python -m pip install --upgrade pip


In [10]:
import grpc

import tensorflow as tf

from tensorflow_serving.apis import predict_pb2
from tensorflow_serving.apis import prediction_service_pb2_grpc


In [11]:
host = 'localhost:8500'

channel = grpc.insecure_channel(host)

stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

In [12]:
stub

In [15]:
from keras_image_helper import create_preprocessor

In [16]:
preprocessor = create_preprocessor('xception', target_size=(299, 299))

In [20]:
url = 'http://bit.ly/mlbookcamp-pants'
X = preprocessor.from_url(url)
X

array([[[[-0.11372548, -0.15294117, -0.19999999],
         [-0.11372548, -0.15294117, -0.19999999],
         [-0.10588235, -0.14509803, -0.19215685],
         ...,
         [-0.01960784, -0.01960784, -0.08235294],
         [-0.04313725, -0.04313725, -0.10588235],
         [-0.11372548, -0.11372548, -0.17647058]],

        [[-0.09019607, -0.12941176, -0.17647058],
         [-0.09019607, -0.12941176, -0.17647058],
         [-0.08235294, -0.12156862, -0.16862744],
         ...,
         [-0.01960784, -0.01960784, -0.08235294],
         [-0.04313725, -0.04313725, -0.10588235],
         [-0.10588235, -0.10588235, -0.16862744]],

        [[-0.09803921, -0.1372549 , -0.18431371],
         [-0.09803921, -0.1372549 , -0.18431371],
         [-0.09019607, -0.12941176, -0.17647058],
         ...,
         [-0.01960784, -0.01960784, -0.08235294],
         [-0.03529412, -0.03529412, -0.09803921],
         [-0.09019607, -0.09019607, -0.15294117]],

        ...,

        [[-0.67058825, -0.7019608 , -0

In [21]:
def np_to_protobuf(data):
    return tf.make_tensor_proto(data, shape=data.shape)

In [24]:
np_to_protobuf(X)

dtype: DT_FLOAT
tensor_shape {
  dim {
    size: 1
  }
  dim {
    size: 299
  }
  dim {
    size: 299
  }
  dim {
    size: 3
  }
}
tensor_content: "\350\350\350\275\234\234\034\276\314\314L\276\350\350\350\275\234\234\034\276\314\314L\276\330\330\330\275\224\224\024\276\304\304D\276\270\270\270\275\204\204\004\276\264\2644\276\210\210\210\275\330\330\330\275\234\234\034\276\360\360p\275\310\310\310\275\224\224\024\276\260\2600\275\250\250\250\275\204\204\004\276\240\240\240\274\360\360p\275\330\330\330\275\300\300@\274\320\320P\275\310\310\310\275\000\201\200;\220\220\020\275\250\250\250\275\000\201\200;\220\220\020\275\250\250\250\275\000\201\200;\220\220\020\275\250\250\250\275\000\201\200;\220\220\020\275\360\360p\275\300\2600=\000\201\200;\240\240\240\274\220\210\210=\000\341\340<\000\201\200;\260\250\250=\300\2600=\300\240\240<\340\330\330=\220\210\210=\300\2600=\360\350\350=\240\230\230=\340\320P=\220\214\014>\320\310\310=\240\230\230=\000\371\370=\260\250\250=\000\361p=\220\21

In [26]:
pb_request = predict_pb2.PredictRequest()

pb_request.model_spec.name = 'clothing-model'
pb_request.model_spec.signature_name = 'serving_default'

pb_request.inputs['input_layer_10'].CopyFrom(np_to_protobuf(X))

In [27]:
pb_request

model_spec {
  name: "clothing-model"
  signature_name: "serving_default"
}
inputs {
  key: "input_layer_10"
  value {
    dtype: DT_FLOAT
    tensor_shape {
      dim {
        size: 1
      }
      dim {
        size: 299
      }
      dim {
        size: 299
      }
      dim {
        size: 3
      }
    }
    tensor_content: "\350\350\350\275\234\234\034\276\314\314L\276\350\350\350\275\234\234\034\276\314\314L\276\330\330\330\275\224\224\024\276\304\304D\276\270\270\270\275\204\204\004\276\264\2644\276\210\210\210\275\330\330\330\275\234\234\034\276\360\360p\275\310\310\310\275\224\224\024\276\260\2600\275\250\250\250\275\204\204\004\276\240\240\240\274\360\360p\275\330\330\330\275\300\300@\274\320\320P\275\310\310\310\275\000\201\200;\220\220\020\275\250\250\250\275\000\201\200;\220\220\020\275\250\250\250\275\000\201\200;\220\220\020\275\250\250\250\275\000\201\200;\220\220\020\275\360\360p\275\300\2600=\000\201\200;\240\240\240\274\220\210\210=\000\341\340<\000\201\200;\260\25

In [28]:
pb_response = stub.Predict(pb_request, timeout=20.0)

In [29]:
pb_response

outputs {
  key: "output_0"
  value {
    dtype: DT_FLOAT
    tensor_shape {
      dim {
        size: 1
      }
      dim {
        size: 10
      }
    }
    float_val: -0.395893335
    float_val: -3.35369921
    float_val: -0.0665492713
    float_val: 0.888122261
    float_val: 7.32219124
    float_val: 0.869614422
    float_val: -1.54651058
    float_val: 3.21258879
    float_val: -0.0861493871
    float_val: -1.88817751
  }
}
model_spec {
  name: "clothing-model"
  version {
    value: 1
  }
  signature_name: "serving_default"
}

In [30]:
preds = pb_response.outputs['output_0'].float_val
preds

[-0.3958933353424072, -3.353699207305908, -0.06654927134513855, 0.8881222605705261, 7.32219123840332, 0.8696144223213196, -1.5465105772018433, 3.2125887870788574, -0.08614938706159592, -1.888177514076233]

In [31]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

In [32]:
dict(zip(classes, preds))

{'dress': -0.3958933353424072,
 'hat': -3.353699207305908,
 'longsleeve': -0.06654927134513855,
 'outwear': 0.8881222605705261,
 'pants': 7.32219123840332,
 'shirt': 0.8696144223213196,
 'shoes': -1.5465105772018433,
 'shorts': 3.2125887870788574,
 'skirt': -0.08614938706159592,
 't-shirt': -1.888177514076233}